# SuperKart - Transformación y enriquecimiento de datos

## Objetivo

Preparar los datos del dataset SuperKart para el análisis
descriptivo de ventas, mediante la validación de su calidad,
la transformación de variables y la creación de nuevos
indicadores que faciliten la interpretación de los resultados
y la toma de decisiones comerciales.

Los datos procesados serán utilizados posteriormente para
la construcción de métricas y visualizaciones en Streamlit.

In [1]:
from datasets import load_dataset
from pathlib import Path
import pandas as pd
import numpy as np

dataset_id = "dev02chandan/sales-forecast-dataset"

dataset = load_dataset(dataset_id)
df_original = dataset["train"].to_pandas()

df = df_original.copy()

print("Dimensiones del dataset:", df.shape)

display(df.head())

Repo card metadata block was not found. Setting CardData to empty.


Dimensiones del dataset: (7414, 12)


,Product_Id,Product_Sugar_Content,Product_Type,Store_Id,Store_Size,Store_Location_City_Type,Store_Type,Product_Weight,Product_Allocated_Area,Product_MRP,Store_Establishment_Year,Product_Store_Sales_Total
0,FD6114,Low Sugar,Frozen Foods,OUT004,Medium,Tier 2,Supermarket Type2,12.66,0.027,117.08,2009.0,2842.40
1,FD5075,Regular,Canned,OUT001,High,Tier 2,Supermarket Type1,14.28,0.031,162.08,1987.0,4130.16
2,FD8233,Low Sugar,Baking Goods,OUT001,High,Tier 2,Supermarket Type1,12.10,0.112,186.31,1987.0,4132.18
3,NC1180,No Sugar,Health and Hygiene,OUT002,Small,Tier 3,Food Mart,9.57,0.010,123.67,1998.0,2279.36
4,FD5680,Low Sugar,Snack Foods,OUT004,Medium,Tier 2,Supermarket Type2,12.03,0.053,113.64,2009.0,2629.15


## Diagnóstico de calidad de los datos

Se revisan los tipos de datos, valores faltantes, registros
duplicados y cantidad de valores únicos por variable,
con el fin de identificar posibles inconsistencias antes
de realizar las transformaciones.

In [2]:
diagnostico = pd.DataFrame({
    "Tipo de dato": df.dtypes.astype(str),
    "Valores nulos": df.isnull().sum(),
    "Valores únicos": df.nunique(),
    "Porcentaje de nulos": (
        df.isnull().mean() * 100
    ).round(2)
})

display(diagnostico)

print("Registros duplicados:", df.duplicated().sum())

display(df.describe().T)

,Tipo de dato,Valores nulos,Valores únicos,Porcentaje de nulos
Product_Id,str,0,7414,0.0
Product_Sugar_Content,str,0,4,0.0
Product_Type,str,0,16,0.0
Store_Id,str,0,3,0.0
Store_Size,str,0,3,0.0
Store_Location_City_Type,str,0,2,0.0
Store_Type,str,0,3,0.0
Product_Weight,float64,0,976,0.0
Product_Allocated_Area,float64,0,223,0.0
Product_MRP,float64,0,5232,0.0


Registros duplicados: 0


,count,mean,std,min,25%,50%,75%,max
Product_Weight,7414.0,12.208025,1.966786,4.000,10.9200,12.260,13.5800,19.820
Product_Allocated_Area,7414.0,0.068814,0.048115,0.004,0.0310,0.056,0.0960,0.297
Product_MRP,7414.0,140.786784,27.355617,31.000,122.6375,141.915,159.0475,226.590
Store_Establishment_Year,7414.0,2002.584570,9.010632,1987.000,1998.0000,2009.000,2009.0000,2009.000
Product_Store_Sales_Total,7414.0,3194.174040,886.408785,33.000,2627.2400,3255.055,3806.0900,5462.860


In [3]:
columnas_categoricas = [
    "Product_Type",
    "Store_Type",
    "Store_Location_City_Type",
    "Store_Size",
    "Product_Sugar_Content"
]

for columna in columnas_categoricas:
    print(f"\n{columna}")
    display(df[columna].value_counts(dropna=False))


Product_Type


Product_Type
Fruits and Vegetables    1067
Snack Foods               963
Frozen Foods              689
Dairy                     651
Household                 633
Baking Goods              617
Canned                    587
Health and Hygiene        539
Meat                      512
Soft Drinks               445
Breads                    166
Hard Drinks               163
Others                    119
Starchy Foods             113
Breakfast                  87
Seafood                    63
Name: count, dtype: int64


Store_Type


Store_Type
Supermarket Type2    4676
Supermarket Type1    1586
Food Mart            1152
Name: count, dtype: int64


Store_Location_City_Type


Store_Location_City_Type
Tier 2    6262
Tier 3    1152
Name: count, dtype: int64


Store_Size


Store_Size
Medium    4676
High      1586
Small     1152
Name: count, dtype: int64


Product_Sugar_Content


Product_Sugar_Content
Low Sugar    4135
Regular      1900
No Sugar     1291
reg            88
Name: count, dtype: int64

In [4]:
columnas_numericas = [
    "Product_Store_Sales_Total",
    "Product_MRP",
    "Product_Weight",
    "Product_Allocated_Area",
    "Store_Establishment_Year"
]

validacion = pd.DataFrame({
    "Mínimo": df[columnas_numericas].min(),
    "Máximo": df[columnas_numericas].max(),
    "Valores negativos": (df[columnas_numericas] < 0).sum(),
    "Valores en cero": (df[columnas_numericas] == 0).sum()
})

display(validacion)

print("Productos únicos:", df["Product_Id"].nunique())
print("Establecimientos únicos:", df["Store_Id"].nunique())

print(
    "Combinaciones producto-establecimiento repetidas:",
    df.duplicated(subset=["Product_Id", "Store_Id"]).sum()
)

,Mínimo,Máximo,Valores negativos,Valores en cero
Product_Store_Sales_Total,33.000,5462.860,0,0
Product_MRP,31.000,226.590,0,0
Product_Weight,4.000,19.820,0,0
Product_Allocated_Area,0.004,0.297,0,0
Store_Establishment_Year,1987.000,2009.000,0,0


Productos únicos: 7414
Establecimientos únicos: 3
Combinaciones producto-establecimiento repetidas: 0


## Limpieza y normalización de datos

Durante la exploración se identificaron 88 registros con
la categoría "reg" en la variable Product_Sugar_Content.

Se unifica esta categoría con "Regular" para mantener
la consistencia de los datos y evitar que una misma
clasificación se analice como dos grupos diferentes.

No se eliminan registros duplicados, debido a que no
se encontraron duplicidades en el diagnóstico inicial.

In [5]:
conteo_antes = df["Product_Sugar_Content"].value_counts()

df["Product_Sugar_Content"] = (
    df["Product_Sugar_Content"]
    .str.strip()
    .replace({"reg": "Regular"})
)

conteo_despues = df["Product_Sugar_Content"].value_counts()

comparacion = pd.DataFrame({
    "Antes": conteo_antes,
    "Después": conteo_despues
}).fillna(0).astype(int)

display(comparacion)

print("Registros originales:", len(df_original))
print("Registros después de la limpieza:", len(df))

assert len(df_original) == len(df)

,Antes,Después
Product_Sugar_Content,,
Low Sugar,4135,4135
No Sugar,1291,1291
Regular,1900,1988
reg,88,0


Registros originales: 7414
Registros después de la limpieza: 7414


In [6]:
print("Valores nulos:", df.isnull().sum().sum())
print("Duplicados:", df.duplicated().sum())

print("\nValidación de variables numéricas:")
display(validacion)

print("\nTipos de establecimiento:")
display(df["Store_Type"].value_counts())

print("\nCategorías de productos:")
display(df["Product_Type"].value_counts())

print("\nTipos de ubicación:")
display(df["Store_Location_City_Type"].value_counts())

print("\nDistribución por establecimiento:")
display(df["Store_Id"].value_counts())

Valores nulos: 0
Duplicados: 0

Validación de variables numéricas:


,Mínimo,Máximo,Valores negativos,Valores en cero
Product_Store_Sales_Total,33.000,5462.860,0,0
Product_MRP,31.000,226.590,0,0
Product_Weight,4.000,19.820,0,0
Product_Allocated_Area,0.004,0.297,0,0
Store_Establishment_Year,1987.000,2009.000,0,0



Tipos de establecimiento:


Store_Type
Supermarket Type2    4676
Supermarket Type1    1586
Food Mart            1152
Name: count, dtype: int64


Categorías de productos:


Product_Type
Fruits and Vegetables    1067
Snack Foods               963
Frozen Foods              689
Dairy                     651
Household                 633
Baking Goods              617
Canned                    587
Health and Hygiene        539
Meat                      512
Soft Drinks               445
Breads                    166
Hard Drinks               163
Others                    119
Starchy Foods             113
Breakfast                  87
Seafood                    63
Name: count, dtype: int64


Tipos de ubicación:


Store_Location_City_Type
Tier 2    6262
Tier 3    1152
Name: count, dtype: int64


Distribución por establecimiento:


Store_Id
OUT004    4676
OUT001    1586
OUT002    1152
Name: count, dtype: int64

## Enriquecimiento de datos

Se crean nuevas variables a partir de los datos existentes
para facilitar la segmentación de productos y el análisis
de su participación en las ventas registradas.

Los rangos de ventas y precios se construyen mediante
cuartiles, utilizando la distribución real de los datos.

También se calcula la participación porcentual de cada
registro respecto al total de ventas del dataset.

Estas variables permiten ampliar el análisis descriptivo
sin modificar los valores originales.

In [7]:
etiquetas = ["Bajo", "Medio", "Alto", "Muy alto"]

df["Sales_Range"], limites_ventas = pd.qcut(
    df["Product_Store_Sales_Total"],
    q=4,
    labels=etiquetas,
    retbins=True
)

df["Price_Range"], limites_precios = pd.qcut(
    df["Product_MRP"],
    q=4,
    labels=etiquetas,
    retbins=True
)

total_ventas = df["Product_Store_Sales_Total"].sum()

df["Sales_Participation_Pct"] = (
    df["Product_Store_Sales_Total"] / total_ventas
) * 100

display(df.head())

,Product_Id,Product_Sugar_Content,Product_Type,Store_Id,Store_Size,Store_Location_City_Type,Store_Type,Product_Weight,Product_Allocated_Area,Product_MRP,Store_Establishment_Year,Product_Store_Sales_Total,Sales_Range,Price_Range,Sales_Participation_Pct
0,FD6114,Low Sugar,Frozen Foods,OUT004,Medium,Tier 2,Supermarket Type2,12.66,0.027,117.08,2009.0,2842.40,Medio,Bajo,0.012003
1,FD5075,Regular,Canned,OUT001,High,Tier 2,Supermarket Type1,14.28,0.031,162.08,1987.0,4130.16,Muy alto,Muy alto,0.017440
2,FD8233,Low Sugar,Baking Goods,OUT001,High,Tier 2,Supermarket Type1,12.10,0.112,186.31,1987.0,4132.18,Muy alto,Muy alto,0.017449
3,NC1180,No Sugar,Health and Hygiene,OUT002,Small,Tier 3,Food Mart,9.57,0.010,123.67,1998.0,2279.36,Bajo,Medio,0.009625
4,FD5680,Low Sugar,Snack Foods,OUT004,Medium,Tier 2,Supermarket Type2,12.03,0.053,113.64,2009.0,2629.15,Medio,Bajo,0.011102


In [8]:
print("Rangos de ventas:")
display(df["Sales_Range"].value_counts().sort_index())

print("\nRangos de precios:")
display(df["Price_Range"].value_counts().sort_index())

print("\nLímites de los rangos de ventas:")
print(limites_ventas)

print("\nLímites de los rangos de precios:")
print(limites_precios)

print("\nParticipación porcentual acumulada:")
print(round(df["Sales_Participation_Pct"].sum(), 6))

print("\nDimensiones originales:", df_original.shape)
print("Dimensiones finales:", df.shape)
print("Valores nulos:", df.isnull().sum().sum())

assert len(df) == len(df_original)

assert np.isclose(
    df["Sales_Participation_Pct"].sum(), 100
)

Rangos de ventas:


Sales_Range
Bajo        1855
Medio       1852
Alto        1853
Muy alto    1854
Name: count, dtype: int64


Rangos de precios:


Price_Range
Bajo        1854
Medio       1853
Alto        1853
Muy alto    1854
Name: count, dtype: int64


Límites de los rangos de ventas:
[  33.    2627.24  3255.055 3806.09  5462.86 ]

Límites de los rangos de precios:
[ 31.     122.6375 141.915  159.0475 226.59  ]

Participación porcentual acumulada:
100.0

Dimensiones originales: (7414, 12)
Dimensiones finales: (7414, 15)
Valores nulos: 0


In [9]:
resumen_establecimientos = df.groupby(
    [
        "Store_Id",
        "Store_Type",
        "Store_Location_City_Type",
        "Store_Size"
    ]
).agg(
    Registros=("Product_Id", "count"),
    Ventas_totales=("Product_Store_Sales_Total", "sum"),
    Ventas_promedio=("Product_Store_Sales_Total", "mean")
).reset_index()

display(resumen_establecimientos.round(2))

,Store_Id,Store_Type,Store_Location_City_Type,Store_Size,Registros,Ventas_totales,Ventas_promedio
0,OUT001,Supermarket Type1,Tier 2,High,1586,6223113.18,3923.78
1,OUT002,Food Mart,Tier 3,Small,1152,2030909.72,1762.94
2,OUT004,Supermarket Type2,Tier 2,Medium,4676,15427583.43,3299.31


## Exportación de los datos procesados

Una vez realizadas las transformaciones y las validaciones
de calidad, se exporta el conjunto de datos procesado en
formato CSV.

El archivo conserva los 7414 registros originales e incorpora
tres nuevas variables derivadas.

Este archivo será utilizado como fuente de información para
el cálculo de métricas y la aplicación de Streamlit,
garantizando la consistencia de los resultados.

In [10]:
proyecto = Path.cwd()

if proyecto.name == "notebooks":
    proyecto = proyecto.parent
elif proyecto.name != "Proyecto-Integrador-EA2":
    proyecto = proyecto / "Proyecto-Integrador-EA2"

carpeta_datos = proyecto / "datos"
carpeta_resultados = proyecto / "resultados"

carpeta_datos.mkdir(parents=True, exist_ok=True)
carpeta_resultados.mkdir(parents=True, exist_ok=True)

ruta_datos = carpeta_datos / "datos_procesados.csv"

df.to_csv(ruta_datos, index=False, encoding="utf-8")

print("Archivo generado correctamente:")
print(ruta_datos)

Archivo generado correctamente:
c:\Users\Usuario\Documents\actividad_1_huggingface\Proyecto-Integrador-EA2\datos\datos_procesados.csv


In [11]:
diagnostico.to_csv(
    carpeta_resultados / "diagnostico_calidad.csv"
)

comparacion.to_csv(
    carpeta_resultados / "normalizacion_azucar.csv"
)

validacion.to_csv(
    carpeta_resultados / "validacion_numerica.csv"
)

resumen_establecimientos.to_csv(
    carpeta_resultados / "resumen_establecimientos.csv",
    index=False
)

rangos = pd.DataFrame({
    "Nivel": etiquetas,
    "Ventas_limite_inferior": limites_ventas[:-1],
    "Ventas_limite_superior": limites_ventas[1:],
    "Precio_limite_inferior": limites_precios[:-1],
    "Precio_limite_superior": limites_precios[1:]
})

rangos.to_csv(
    carpeta_resultados / "rangos_cuartiles.csv",
    index=False
)

print("Evidencias guardadas correctamente")

Evidencias guardadas correctamente


In [12]:
df_verificacion = pd.read_csv(ruta_datos)

print("Dimensiones:", df_verificacion.shape)
print("Valores nulos:", df_verificacion.isnull().sum().sum())
print("Duplicados:", df_verificacion.duplicated().sum())

assert df_verificacion.shape == (7414, 15)

assert np.isclose(
    df_verificacion["Product_Store_Sales_Total"].sum(),
    total_ventas
)

print("Ventas totales:", round(total_ventas, 2))
print("Validación exitosa")

Dimensiones: (7414, 15)
Valores nulos: 0
Duplicados: 0
Ventas totales: 23681606.33
Validación exitosa
